# CHI-Bench Marathon Workflow Analysis

This notebook analyzes the three Marathon session definitions in CHI-Bench.

The analysis focuses on session composition, underlying task coverage, and preserved workflow starting states. It uses the public session manifests and does not inspect hidden evaluation artifacts, test expectations, or solution files.

## Objective

The Marathon family evaluates long-horizon execution by grouping the underlying workflow tasks into three extended sessions:

- Prior Authorization — Provider
- Prior Authorization — Payer / Utilization Management
- Care Management

The three sessions collectively reference 75 underlying task instances. These are not additional unique clinical cases; they correspond to workflows already represented in the benchmark's underlying task families.

In [1]:
from pathlib import Path
import csv
from collections import Counter

DATA_FILE = Path("analysis/data/marathon_task_metadata.csv")

rows = []

with DATA_FILE.open("r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    rows = list(reader)

print(f"Marathon task instances: {len(rows)}")

Marathon task instances: 75


## Session Composition

Each Marathon session contains 25 underlying workflow tasks.

In [2]:
session_counts = Counter(row["session_id"] for row in rows)

for session, count in session_counts.items():
    print(f"{session}: {count}")

print(f"Total underlying task instances: {sum(session_counts.values())}")

prior_auth_provider_session_v1: 25
prior_auth_um_session_v1: 25
care_management_session_v1: 25
Total underlying task instances: 75


## Domain Coverage

The three Marathon sessions cover the complete 25-task sets for provider prior authorization, payer utilization management, and care management.

In [3]:
domain_counts = Counter(row["domain"] for row in rows)

for domain, count in domain_counts.items():
    print(f"{domain}: {count}")

prior_auth_provider: 25
prior_auth_um: 25
care_management: 25


## PA-UM Starting-State Distribution

The payer Marathon session preserves the starting state of each underlying utilization-management workflow.

The five observed starting states are intake, triage, nurse review, medical review, and peer-to-peer.

In [4]:
um_rows = [row for row in rows if row["domain"] == "prior_auth_um"]

initial_states = Counter(
    row["initial_state"]
    for row in um_rows
    if row["initial_state"]
)

for state, count in initial_states.items():
    print(f"{state}: {count}")

md_review: 4
intake: 5
triage: 4
nurse_review: 8
p2p: 4


## Key Observations

- The Marathon family contains 3 long-horizon session definitions.
- Each session references 25 underlying workflows.
- The sessions collectively cover 75 underlying task instances.
- Provider, payer/UM, and care-management sessions each cover their complete 25-task workflow set.
- The payer/UM session preserves five distinct workflow starting states.
- Nurse review is the largest PA-UM starting-state group, representing 8 of 25 workflows (32%).
- The 75 underlying task instances should not be counted as 75 additional unique clinical cases because the Marathon sessions orchestrate workflows already represented elsewhere in the benchmark.

## Portfolio Interpretation

The Marathon family adds a different evaluation dimension from the individual workflow families. Rather than introducing a new clinical domain, it tests whether an agent can maintain reliable execution across a much longer sequence of healthcare workflows.

For healthcare analytics and automation, this is relevant to operational settings where an AI system must process multiple cases in one sustained session while preserving task context and workflow state.

## Limitations

This analysis describes benchmark structure rather than real-world healthcare utilization.

The 75 underlying task instances represent references within three Marathon sessions and should not be interpreted as 75 unique patients, claims, or clinical cases.

The analysis is based on the public session manifests and task metadata available in the repository. It does not inspect hidden evaluation artifacts, solution files, or test expectations.

## Conclusion

CHI-Bench Marathon extends the benchmark from individual workflow execution to long-horizon orchestration.

Across three sessions, the Marathon family covers the complete 25-task sets for provider prior authorization, payer utilization management, and care management. The PA-UM session additionally preserves stage-specific starting states, demonstrating that the Marathon layer retains meaningful workflow context across an extended execution sequence.

This makes Marathon an important complement to the task-level analyses in this portfolio: it shifts the analytical perspective from individual workflow complexity toward sustained multi-workflow execution.